# Notebook 05 — Feature-Based Ensemble

Build a Poisson ensemble on top of the Dixon-Coles baseline (RPS 0.1646).
Strategy: engineer leakage-safe features, train gradient-boosted Poisson learners, stack with a penalized meta.

In [59]:
import sys
sys.path.append('..')   # so `src` is importable from notebooks/

import pandas as pd 
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from scipy.stats import poisson
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor

# User-defined functions 
from src.evaluation import ranked_probability_score

## Load data

In [2]:
results = pd.read_parquet('../data/interim/results_clean.parquet')
print(results.head())
print(results.shape)

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
(49482, 9)


In [3]:
# Same data scope + split as Model 1, so both models are evaluated on the identical eval set
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

# Unique match key so per-team merges stay one-to-one (a team can play twice on the same date,
# which makes (date, team) non-unique and fans out the merges — match_id prevents that).
# insert at position 0 so it's the leading column.
model_df.insert(0, 'match_id', model_df.index)

## Feature 1 — As-of Elo rating

Run Elo chronologically and record each team's rating **before** the match update.
No leakage: the rating reflects only prior results. Tuned params from notebook 04: K=30, base=1500.

In [4]:
# As-of Elo feature: run Elo chronologically, record each team's rating BEFORE the match
def add_elo_features(matches, k=30, base_rating=1500):
    # Initialize ratings and elo lists
    ratings = {}
    elo_home_list = []
    elo_away_list = []

    for _, row in matches.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']

        # Get current ratings or initialize them
        elo_home = ratings.get(home_team, base_rating)
        elo_away = ratings.get(away_team, base_rating)

        elo_home_list.append(elo_home)
        elo_away_list.append(elo_away)

        # Calculate expected 
        expected_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))
        expected_away = 1 - expected_home

        # Determine actual result
        if row['home_score'] > row['away_score']:
            actual_home = 1
            actual_away = 0
        elif row['home_score'] == row['away_score']:
            actual_home = 0.5
            actual_away = 0.5
        else:
            actual_home = 0
            actual_away = 1

        # Add change
        change = k * (actual_home - expected_home)
        ratings[home_team] = elo_home + change
        ratings[away_team] = elo_away - change

    matches = matches.copy()
    matches['elo_home'] = elo_home_list
    matches['elo_away'] = elo_away_list
    
    return matches

model_df = add_elo_features(model_df)   # model_df must be chronologically sorted (it is)
print(model_df[['date', 'home_team', 'away_team', 'elo_home', 'elo_away']].head())

        date home_team       away_team  elo_home  elo_away
0 1970-01-04     Malta      Luxembourg    1500.0    1500.0
1 1970-01-14   England     Netherlands    1500.0    1500.0
2 1970-01-28    Israel     Netherlands    1500.0    1500.0
3 1970-02-04      Peru  Czechoslovakia    1500.0    1500.0
4 1970-02-06  Cameroon     Ivory Coast    1500.0    1500.0


## Feature 2 — Rolling form (goals scored / conceded)

Trailing 5-match rolling mean of goals scored and conceded per team.
Shifted by 1 (`shift(1).rolling(5).mean()`) so each row uses only prior matches — no leakage.
Computed on a long-format (one row per team per match) table, then merged back as home/away columns.

In [5]:
# Reshape to a per-team-per-match table for computing rolling form
def reshape_to_long_format(df):
    home_rows = df.rename(columns={
        'home_team': 'team',
        'home_score': 'goals_scored',
        'away_score': 'goals_conceded'
    })[['match_id', 'date', 'team', 'goals_scored', 'goals_conceded']]

    away_rows = df.rename(columns={
        'away_team': 'team',
        'away_score': 'goals_scored',
        'home_score': 'goals_conceded'
    })[['match_id', 'date', 'team', 'goals_scored', 'goals_conceded']]

    long_format = pd.concat([home_rows, away_rows], ignore_index=True)
    return long_format

long_df = reshape_to_long_format(model_df)
print(long_df.head())

   match_id       date      team  goals_scored  goals_conceded
0         0 1970-01-04     Malta             1               1
1         1 1970-01-14   England             0               0
2         2 1970-01-28    Israel             0               1
3         3 1970-02-04      Peru             0               2
4         4 1970-02-06  Cameroon             3               2


In [6]:
# Sort the long df by team and then by date
long_df = long_df.sort_values(by=['team', 'date'])

# Create trailing rolling mean
N = 5 
form = long_df.groupby('team')[['goals_scored', 'goals_conceded']].transform(lambda s: s.shift(1).rolling(N).mean())
long_df['form_scored'] = form['goals_scored']
long_df['form_conceded'] = form['goals_conceded']

# print(long_df) - Successful

# Merge form back to model_df 2 times, keyed on (match_id, team) so it's strictly one-to-one
# Home form 
home_form = long_df[['match_id', 'team', 'form_scored', 'form_conceded']].rename(columns={
    'team': 'home_team',
    'form_scored': 'home_form_scored',
    'form_conceded': 'home_form_conceded'
})

model_df = model_df.merge(home_form, on=['match_id', 'home_team'], how='left')

# Away form
away_form = long_df[['match_id', 'team', 'form_scored', 'form_conceded']].rename(columns={
    'team': 'away_team',
    'form_scored': 'away_form_scored',
    'form_conceded': 'away_form_conceded'
})

model_df = model_df.merge(away_form, on=['match_id', 'away_team'], how='left')

## Feature 3 — Rest days

Days since each team's previous match. Computed via `groupby('team')['date'].diff()` on the long table — naturally backward-looking, no leakage. NaN for a team's first appearance.

In [7]:
# Create rest days 
long_df['rest_days'] = long_df.groupby('team')['date'].diff().dt.days

# Merge back to model_df, keyed on (match_id, team) so it's strictly one-to-one
# Home rest days
home_rest_days = long_df[['match_id', 'team', 'rest_days']].rename(columns={
    'team': 'home_team',
    'rest_days': 'home_rest_days'
})

model_df = model_df.merge(home_rest_days, on=['match_id', 'home_team'], how='left')

# Away rest days
away_rest_days = long_df[['match_id', 'team', 'rest_days']].rename(columns={
    'team': 'away_team',
    'rest_days': 'away_rest_days'
})

model_df = model_df.merge(away_rest_days, on=['match_id', 'away_team'], how='left')

## Feature 4 — Tournament tier (ordinal)

Ordinal encoding of match importance: Friendly=0, Qualifier=1, Continental/regional cup=2, FIFA World Cup=3.
String matching on the `tournament` column — `"qualification"` catches all qualifiers, exact match for Friendly and FIFA World Cup, everything else falls to tier 2.

In [8]:
# Create a tournament tier 
def tournament_tier(name):
    if name == 'Friendly':
        return 0
    elif 'qualification' in name.lower():
        return 1
    elif name == 'FIFA World Cup':
        return 3
    else:
        return 2 # everything else = continental/regional Cup
    
model_df['tournament_tier'] = model_df['tournament'].apply(tournament_tier)

## Feature 5 — Head-to-head (matches played + goal difference)

Two features capturing the specific pairing's history, beyond what team-level Elo says:
- **`h2h_matches_played`** — count of prior meetings (the gate: tells the booster how much to trust the goal-diff).
- **`h2h_goal_diff`** — average goal difference across prior meetings, from the current home team's perspective. `NaN` if they've never met.

A match-up is symmetric but goal difference is directional, so we anchor on a **reference team** = the alphabetically-first of the pair (`key = tuple(sorted([home, away]))`). All history is stored in reference-team terms; we flip the sign twice — once on **read** (reference → home perspective) and once on **write** (home → reference perspective). Leakage-safe: features are appended *before* the current match updates the running dicts (same read→append→update rhythm as the Elo loop).

In [9]:
def add_h2h_features(matches):
    # Initialize count and goal difference
    h2h_matches_played = []
    h2h_goal_diff = []

    gd_sum = {} # cumulative goal diff from reference team's perspective
    count = {} # number of prior meetings

    for _, row in matches.iterrows():
        key = tuple(sorted([row['home_team'], row['away_team']]))
        reference_team = key[0]
        
        # Read the current state (before this match)
        prior_count = count.get(key, 0)
        prior_gd = gd_sum.get(key, 0) # reference team's perspective

        # Convert to home team's perspective and compute the feature
        if row['home_team'] == reference_team:
            home_perspective_gd = prior_gd
        else: 
            home_perspective_gd = -prior_gd

        if prior_count > 0:
            avg_gd = home_perspective_gd / prior_count
        else:
            avg_gd = np.nan

        # Append to lists
        h2h_matches_played.append(prior_count)
        h2h_goal_diff.append(avg_gd)

        # Update the dicts with this match's results (after the read, so no leakage)
        match_gd = row['home_score'] - row['away_score']
        if row['home_team'] != reference_team:
            match_gd = -match_gd
        
        gd_sum[key] = prior_gd + match_gd
        count[key] = prior_count + 1

    matches = matches.copy()
    matches['h2h_matches_played'] = h2h_matches_played
    matches['h2h_goal_diff'] = h2h_goal_diff
    return matches

model_df = add_h2h_features(model_df)

## Train / eval / WC2026 split

Placed **after** all feature engineering on purpose — every feature (Elo, rolling form, rest days, tournament tier, H2H) is computed on the full chronological `model_df` first, so cross-slice history is available. Splitting earlier would truncate a team's history and leak the split boundary into the feature values.

Same boundaries as notebooks 03 (Dixon-Coles) and 04 (Elo) so the ensemble is judged on the identical eval set:
- **train** — before 2024-01-01 (fit the boosters here)
- **eval** — 2024-01-01 → pre-WC-2026, WC excluded (tune + select models here)
- **wc2026** — the tournament itself, held aside; the sole test set

Boundaries are mutually exclusive so no match lands in two slices.

In [10]:
# Leakage-safe split — boundaries mutually exclusive so no match is in two slices
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']                        # eval model sees only this
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]       # held-out test set (WC excluded)
wc2026_mask_df = model_df[wc2026_mask]                                      # forecast target, held aside

## Modeling approach — why almost no preprocessing

The base learners are **gradient-boosted trees** (XGBoost, LightGBM, CatBoost, HistGB), all with a Poisson objective. That choice removes most of the usual preprocessing pipeline:

- **No PolynomialFeatures / interaction terms.** Trees build interactions natively by splitting on one feature within a region defined by another. Explicit polynomial terms are pure redundancy.
- **No log transforms.** The Poisson objective already models `log(λ)` internally (log link), and trees are invariant to any monotonic transform of an input — they split on rank order, so `log(x)` and `x` give identical splits. Log-transforming inputs cannot change the fit.
- **No scaling / standardization.** Trees are scale-invariant; splits depend on order, not magnitude.
- **No imputation.** XGBoost, LightGBM, CatBoost and HistGB all handle `NaN` natively, learning a default split direction for missing values. The form/rest-days/H2H NaNs feed in untouched — the missingness is itself informative (a debut team, a never-met pairing).

So features go into the boosters **raw**.

### Feature selection — no RFE

We keep all engineered features and do **not** run RFE (recursive feature elimination):

1. The feature set is already small (~11) and hand-curated against the data ceiling — there is nothing to prune.
2. RFE is unstable with trees: correlated features (e.g. `elo_home`/`elo_away`, or Elo vs form, which both encode strength) split their importance arbitrarily, so RFE can drop a useful feature whose twin happened to absorb the credit that round. It's also expensive (N refits × CV).
3. Trees self-select — a useless feature simply never gets split on — and regularization handles the rest.

When we genuinely want to know whether a feature earns its keep (e.g. `tournament_tier`, `h2h_*`), we run a **targeted ablation**: drop it, refit, re-score **eval RPS**. Asking the metric we care about is more honest and decision-relevant than trusting a noisy importance score. Post-fit importances are used only as a **sanity diagnostic** (does Elo/form dominate?), never as a pruning trigger.

## Reshape to attacker/defender long format

Each match becomes **two rows** — one per scoring side — so a single Poisson model learns attack *and* defense at once, and the encoding is symmetric by construction (every team is seen as both attacker and defender). This is the ensemble analogue of the Dixon-Coles `reshape_to_long`, but carrying the engineered strength/form features instead of team dummies.

- **home-attacker row**: home team's features → `attacker_*`, away team's → `defender_*`, target `goals = home_score`, `is_home = 1` unless neutral.
- **away-attacker row**: the mirror — away team's features → `attacker_*`, target `goals = away_score`, `is_home = 0` (the away side never gets home advantage).

`tournament_tier`, `neutral`, and `h2h_matches_played` are **match-level** — identical in both rows. `h2h_goal_diff` is the one directional feature: it was stored in the home team's perspective, so it's **sign-flipped** on the away-attacker row. Doubles the training rows (38,900 → 77,800) — real leverage against the data ceiling.

In [20]:
# Create a function to reshape to long format for training
def reshape_to_long_training(df):
    # 1st subframe, home_attacker_rows
    home_attacker_rows = df.rename(columns={
        'elo_home': 'attacker_elo',
        'elo_away': 'defender_elo',
        'home_form_scored': 'attacker_form_scored',
        'home_form_conceded': 'attacker_form_conceded',
        'away_form_scored': 'defender_form_scored',
        'away_form_conceded': 'defender_form_conceded',
        'home_rest_days': 'attacker_rest_days',
        'away_rest_days': 'defender_rest_days',
        'home_score': 'goals'
    })[['match_id', 'date', 'attacker_elo', 'defender_elo', 'attacker_form_scored', 'attacker_form_conceded',
        'defender_form_scored', 'defender_form_conceded', 'attacker_rest_days', 'defender_rest_days',
        'tournament_tier', 'neutral', 'h2h_matches_played', 'h2h_goal_diff', 'goals']].copy()
    
    home_attacker_rows['is_home'] = (~home_attacker_rows['neutral']).astype(int)

    # 2nd subframe, away_attacker_rows — away team is the attacker, so every home/away pair swaps
    away_attacker_rows = df.rename(columns={
        'elo_away': 'attacker_elo',
        'elo_home': 'defender_elo',
        'away_form_scored': 'attacker_form_scored',
        'away_form_conceded': 'attacker_form_conceded',
        'home_form_scored': 'defender_form_scored',
        'home_form_conceded': 'defender_form_conceded',
        'away_rest_days': 'attacker_rest_days',
        'home_rest_days': 'defender_rest_days',
        'away_score': 'goals'
    })[['match_id', 'date', 'attacker_elo', 'defender_elo', 'attacker_form_scored', 'attacker_form_conceded',
        'defender_form_scored', 'defender_form_conceded', 'attacker_rest_days', 'defender_rest_days',
        'tournament_tier', 'neutral', 'h2h_matches_played', 'h2h_goal_diff', 'goals']].copy()
    
    away_attacker_rows['is_home'] = 0
    # h2h_goal_diff was stored in the home team's perspective; flip sign now that the attacker is the away team
    away_attacker_rows['h2h_goal_diff'] = -away_attacker_rows['h2h_goal_diff']

    return pd.concat([home_attacker_rows, away_attacker_rows], ignore_index=True)

long_train = reshape_to_long_training(train_df)

In [27]:
features = [
    'attacker_elo', 'defender_elo',
    'attacker_form_scored', 'attacker_form_conceded',
    'defender_form_scored', 'defender_form_conceded',
    'attacker_rest_days', 'defender_rest_days',
    'tournament_tier', 'neutral',
    'h2h_matches_played', 'h2h_goal_diff', 'is_home'
]

X_train = long_train[features]
y_train = long_train['goals']

In [39]:
print(eval_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2543 entries, 38900 to 41442
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   match_id            2543 non-null   int64         
 1   date                2543 non-null   datetime64[us]
 2   home_team           2543 non-null   str           
 3   away_team           2543 non-null   str           
 4   home_score          2543 non-null   int32         
 5   away_score          2543 non-null   int32         
 6   tournament          2543 non-null   str           
 7   city                2543 non-null   str           
 8   country             2543 non-null   str           
 9   neutral             2543 non-null   bool          
 10  elo_home            2543 non-null   float64       
 11  elo_away            2543 non-null   float64       
 12  home_form_scored    2542 non-null   float64       
 13  home_form_conceded  2542 non-null   float64       
 14

In [47]:
# Create evaluate_booster function
def evaluate_booster(model, df, max_goals=10):
    # Reshape eval_df to long format
    long_eval = reshape_to_long_training(df)

    # Predict with model
    preds = model.predict(long_eval[features])

    n = len(df)
    lambda_home = preds[:n]
    lambda_away = preds[n:]

    home_score = df['home_score'].to_numpy()
    away_score = df['away_score'].to_numpy()

    rps_values = []
    for i in range(n):
        # Get home and away probs
        home_probs = poisson.pmf(np.arange(max_goals + 1), lambda_home[i])
        away_probs = poisson.pmf(np.arange(max_goals + 1), lambda_away[i])

        # Create matrix 
        matrix = np.outer(home_probs, away_probs)
        probs = [
            np.tril(matrix, -1).sum(),
            np.trace(matrix),
            np.triu(matrix, 1).sum()
        ]

        # One-Hot actual outcomes
        if home_score[i] > away_score[i]:
            actual = [1, 0, 0]
        elif home_score[i] == away_score[i]:
            actual = [0, 1, 0]
        else:
            actual = [0, 0, 1]

        value = ranked_probability_score(probs, actual)
        rps_values.append(value)

    # Mean rps
    mean_rps = np.mean(rps_values)
    return mean_rps

In [54]:
# Create XGBRegressor
xgb_model = xgb.XGBRegressor(objective='count:poisson', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_rps = evaluate_booster(xgb_model, eval_df)
print(f'XGB RPS: {xgb_rps:.4f}')

# Create LGBMRegressor
lgb_model = lgb.LGBMRegressor(objective='poisson', random_state=42, verbose=0)
lgb_model.fit(X_train, y_train)
lgb_rps = evaluate_booster(lgb_model, eval_df)
print(f'LGB RPS: {lgb_rps:.4f}')

# Create CatBoostRegressor
cb_model = cb.CatBoostRegressor(loss_function='Poisson', random_state=42, verbose=0)
cb_model.fit(X_train, y_train)
cb_rps = evaluate_booster(cb_model, eval_df)
print(f'CB RPS: {cb_rps:.4f}')

# Create HistGradientBoostingRegressor
hgb_model = HistGradientBoostingRegressor(loss='poisson', random_state=42)
hgb_model.fit(X_train, y_train)
hgb_rps = evaluate_booster(hgb_model, eval_df)
print(f'HGB RPS: {hgb_rps:.4f}')

# Create RandomForestRegressor
rf_model = RandomForestRegressor(criterion='poisson', random_state=42)
rf_model.fit(X_train, y_train)
rf_rps = evaluate_booster(rf_model, eval_df)
print(f'RF RPS: {rf_rps:.4f}')

XGB RPS: 0.1690
LGB RPS: 0.1689
CB RPS: 0.1683
HGB RPS: 0.1697
RF RPS: 0.1730


In [61]:
# Add PoissonRegressor as well 
numeric_features = [
    'attacker_elo', 'defender_elo',
    'attacker_form_scored', 'attacker_form_conceded',
    'defender_form_scored', 'defender_form_conceded',
    'attacker_rest_days', 'defender_rest_days',
    'h2h_matches_played', 'h2h_goal_diff',
]

numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

preprocessor = ColumnTransformer(
    [
        ('num', numeric_pipeline, numeric_features),
        ('tier', OneHotEncoder(handle_unknown='ignore'), ['tournament_tier']),
        # neutral and is_home are passed through (already 0/1)
    ], remainder='passthrough')

glm_model = Pipeline([
    ('prep', preprocessor),
    ('glm', PoissonRegressor(alpha=1.0, max_iter=1000))
])

glm_model.fit(X_train, y_train)
glm_rps = evaluate_booster(glm_model, eval_df) # Same evaluate_booster as before (Never the name)
print(f'GLM RPS: {glm_rps:.4f}')

GLM RPS: 0.1789
